In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

<details>
<summary><b>Versioni dei pacchetti</b></summary>

Il codice in questa pagina è stato sviluppato usando i seguenti requisiti.
Si consiglia di usare queste versioni o versioni più recenti.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Oltre a [visualizzare le istruzioni su un circuito](/guides/visualize-circuits), potresti voler visualizzare la pianificazione (scheduling) di un circuito usando il metodo [`timeline_drawer`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.timeline_drawer) di Qiskit. Questa visualizzazione può aiutarti, ad esempio, a individuare rapidamente i tempi di inattività sui qubit. Tuttavia, questo metodo non restituisce risultati accurati per i circuiti dinamici. Per visualizzare lo scheduling dei circuiti dinamici, usa il metodo `draw_circuit_schedule_timing`, come descritto nella sezione [Supporto di Qiskit Runtime](#qr-support).

## Esempi

Per visualizzare un circuito pianificato, puoi chiamare questa funzione con un insieme di argomenti di controllo. La maggior parte dell'aspetto dell'immagine di output può essere modificata tramite un foglio di stile, ma non è obbligatorio.

### Disegna con il foglio di stile predefinito

In [1]:
from qiskit import QuantumCircuit
from qiskit.visualization.timeline import draw
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import generate_preset_pass_manager

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

backend = GenericBackendV2(5)

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

draw(isa_circuit, target=backend.target)

<Image src="../docs/images/guides/visualize-circuit-timing/extracted-outputs/5b21b8c1-0.svg" alt="Output of the previous code cell" />

![Output della cella di codice precedente](../docs/images/guides/visualize-circuit-timing/extracted-outputs/5b21b8c1-0.svg)

### Disegna con un foglio di stile adatto al debug del programma

In [2]:
from qiskit import QuantumCircuit
from qiskit.visualization.timeline import draw, IQXDebugging
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import generate_preset_pass_manager

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

backend = GenericBackendV2(5)
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)
draw(isa_circuit, style=IQXDebugging(), target=backend.target)

<Image src="../docs/images/guides/visualize-circuit-timing/extracted-outputs/907dc46c-0.svg" alt="Output of the previous code cell" />

![Output della cella di codice precedente](../docs/images/guides/visualize-circuit-timing/extracted-outputs/907dc46c-0.svg)

Puoi creare funzioni di generazione o di layout personalizzate e aggiornare un foglio di stile esistente con le funzioni personalizzate. In questo modo puoi controllare la maggior parte dell'aspetto dell'immagine di output senza modificare il codice sorgente del drawer dei circuiti pianificati. Consulta il riferimento API di [`timeline_drawer`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.timeline_drawer) per altri esempi.
<span id="qr-support"></span>

## Supporto di Qiskit Runtime

Sebbene il timeline drawer integrato in Qiskit sia utile per i circuiti statici, potrebbe non riflettere con precisione il timing dei [circuiti dinamici](/guides/classical-feedforward-and-control-flow) a causa di operazioni implicite come la diffusione (broadcasting) e la determinazione dei rami. Come parte del supporto per i circuiti dinamici, Qiskit Runtime restituisce informazioni accurate sul timing del circuito all'interno dei risultati del job, quando richiesto.

> **Note:** - Questa è una funzione sperimentale. È in stato di anteprima ed è quindi soggetta a modifiche.
> - Questa funzione si applica solo ai job del Sampler di Qiskit Runtime.
> - Sebbene il tempo totale del circuito sia restituito nei metadati di "compilation", questo NON è il tempo usato per la fatturazione (quantum time).

### Abilitare il recupero dei dati di timing

Per abilitare il recupero dei dati di timing, imposta il flag sperimentale `scheduler_timing` su `True` durante l'esecuzione del job con il primitivo.

In [3]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

sampler = SamplerV2(backend)
sampler.options.experimental = {
    "execution": {
        "scheduler_timing": True,
    },
}

sampler_job = sampler.run([isa_circuit])
result = sampler_job.result()

### Accedere ai dati di timing del circuito
Quando richiesto, i dati di timing del circuito per ogni PUB vengono restituiti nei metadati del risultato del job, sotto `["compilation"]["scheduler_timing"]["timing"]`. Questo campo contiene le informazioni grezze sul timing. Per visualizzare le informazioni sul timing, usa lo strumento di visualizzazione integrato, come descritto nella sezione [Visualizzare i timing](#visualize-timings).

Usa il codice seguente per accedere ai dati di timing del circuito per il primo PUB:

In [4]:
job_result = sampler_job.result()
circuit_schedule = job_result[0].metadata["compilation"]["scheduler_timing"]
circuit_schedule_timing = circuit_schedule["timing"]

#### Comprendere i dati di timing grezzi
Sebbene il caso d'uso più comune sia visualizzare i dati di timing del circuito tramite il metodo `draw_circuit_schedule_timing`, può essere utile comprendere la struttura dei dati di timing grezzi restituiti. Questo può aiutarti, ad esempio, a estrarre informazioni in modo programmatico.

I dati di timing restituiti in `["compilation"]["scheduler_timing"]["timing"]` sono una lista di stringhe. Ogni stringa rappresenta una singola istruzione su un canale ed è separata da virgole nei seguenti tipi di dati:

- `Branch` - Determina se l'istruzione si trova in un flusso di controllo (then / else) o in un ramo principale.
- `Instruction` - Il gate e il qubit su cui operare.
- `Channel` - Il canale a cui viene assegnata l'istruzione. Può essere uno dei seguenti:
      - `Qubit x` - Il canale di drive per il qubit _x_.
      - `AWGRx_y` (generatore di forma d'onda arbitraria per la lettura) - Usato dai canali di readout per comunicare durante la misurazione dei qubit. Gli argomenti _x_ e _y_ corrispondono rispettivamente all'ID dello strumento di readout e al numero del qubit.
- `T0` - Il tempo di inizio dell'istruzione all'interno della pianificazione completa.
- `Duration` - La durata dell'istruzione, in unità di _dt_ secondi, dove 1 dt = 1 ciclo di scheduling. Puoi trovare il valore `dt` di un backend usando [`backend.dt`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.BackendV2#dt).
- `Pulse` - Il tipo di operazione a impulso utilizzata.

Esempio:

In [5]:
from qiskit_ibm_runtime.visualization import draw_circuit_schedule_timing

# Create a figure from the metadata
fig = draw_circuit_schedule_timing(
    circuit_schedule=circuit_schedule_timing,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)

# Uncomment the following line to display the figure
# fig.show(renderer="notebook")

# Save to a file
# fig.write_html("scheduler_timing.html")

<span id="visualize-timings"></span>
### Visualizzare i timing
Con `qiskit-ibm-runtime` v0.43.0 o versioni successive, puoi visualizzare i timing dei circuiti. Per visualizzare i timing, devi prima convertire i metadati del risultato in `fig` usando il [metodo `draw_circuit_schedule_timing`.](https://github.com/Qiskit/qiskit-ibm-runtime/blob/3d1bf1e1d49e5123841639fce259859c90ce9314/qiskit_ibm_runtime/visualization/draw_circuit_schedule_timings.py#L26) Questo metodo restituisce una figura `plotly`, che puoi visualizzare direttamente, salvare in un file, o entrambe le cose. Per ulteriori informazioni sui comandi `plotly` da usare, consulta [`fig.show()`](https://plotly.com/python-api-reference/generated/plotly.io.show.html) e [`fig.write_image("<path.format>")`.](https://plotly.com/python-api-reference/generated/plotly.io.write_image.html)

In [6]:
from qiskit_ibm_runtime import SamplerV2, QiskitRuntimeService
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Create a dynamic circuit

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
qc = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

qc.h(q0)
qc.measure(q0, c0)
with qc.if_test((c0, 1)):
    qc.x(q0)
qc.measure(q0, c0)

# Convert to an ISA circuit for the given backend

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

# Generate samplers for backend targets
sampler = SamplerV2(backend)
sampler.options.experimental = {"execution": {"scheduler_timing": True}}

# Submit jobs
sampler_job = sampler.run([isa_circuit])
result = sampler_job.result()

print(
    f">>> {' Job ID:':<10}  {sampler_job.job_id()} ({sampler_job.status()})"
)

>>>  Job ID:    d5kk3cn853es738e01dg (DONE)


![Passando il mouse sull'output vengono mostrate informazioni come l'inizio, la fine e la durata.](../docs/images/guides/visualize-circuit-timing/image_1.svg 'Esempio di una figura generata')
#### Comprendere la figura generata
L'immagine dei dati di timing del circuito prodotta da `draw_circuit_schedule_timing` trasmette le seguenti informazioni:

- L'asse X è il tempo in unità di _dt_ secondi, dove 1 dt = 1 ciclo di scheduling. Puoi trovare il valore `dt` di un backend usando [`backend.dt`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.BackendV2#dt).
- L'asse Y è il canale (i canali vanno intesi come strumenti che emettono impulsi).
    - `Receive channel` - L'unico canale che non è uno strumento di per sé. È un'istruzione eseguita su tutti i canali che fanno parte di una procedura di comunicazione con l'hub in quel momento.
    - `Qubit x` - Il canale di drive per il qubit x.
    - `AWGRx_y` (generatore di forma d'onda arbitraria per la lettura) - Usato dai canali di readout per comunicare durante la misurazione dei qubit. Gli argomenti _x_ e _y_ corrispondono rispettivamente all'ID dello strumento di readout e al numero del qubit.
    - `Hub` - Controlla la diffusione (broadcasting).

Inoltre, ogni istruzione ha il formato *X_Y*, dove *X* è il nome dell'istruzione e *Y* è il tipo di impulso. Un tipo `play` applica impulsi di controllo, mentre un tipo `capture` registra lo stato del qubit. Puoi anche passare il mouse su ogni istruzione per ottenere maggiori dettagli. Ad esempio, la figura precedente mostra un impulso di controllo per il gate X applicato al qubit 10 a 1161 dt.
### Esempio end-to-end
Questo esempio ti mostra come abilitare l'opzione, ottenere le informazioni di timing del circuito dai metadati e visualizzarle come immagine.

Per prima cosa, configura l'ambiente, definisci i circuiti e convertili in circuiti ISA, quindi definisci ed esegui i job.

In [7]:
# Get the circuit schedule timing
result[0].metadata["compilation"]["scheduler_timing"]["timing"]

'main,rz_0,Qubit 0,929,0,shift_phase\nmain,sx_0,Qubit 0,929,8,play\nmain,sx_0,Qubit 0,933,0,shift_phase\nmain,rz_0,Qubit 0,937,0,shift_phase\nmain,barrier,Qubit 0,937,0,barrier\nmain,barrier,Qubit 0,937,0,barrier\nmain,barrier,Qubit 1,937,0,barrier\nmain,barrier,Qubit 2,937,0,barrier\nmain,barrier,Qubit 3,937,0,barrier\nmain,barrier,Qubit 4,937,0,barrier\nmain,barrier,Qubit 5,937,0,barrier\nmain,barrier,Qubit 6,937,0,barrier\nmain,barrier,Qubit 7,937,0,barrier\nmain,barrier,Qubit 8,937,0,barrier\nmain,barrier,Qubit 9,937,0,barrier\nmain,barrier,Qubit 10,937,0,barrier\nmain,barrier,Qubit 11,937,0,barrier\nmain,barrier,Qubit 12,937,0,barrier\nmain,barrier,Qubit 13,937,0,barrier\nmain,barrier,Qubit 14,937,0,barrier\nmain,barrier,Qubit 15,937,0,barrier\nmain,barrier,Qubit 16,937,0,barrier\nmain,barrier,Qubit 17,937,0,barrier\nmain,barrier,Qubit 18,937,0,barrier\nmain,barrier,Qubit 19,937,0,barrier\nmain,barrier,Qubit 20,937,0,barrier\nmain,barrier,Qubit 21,937,0,barrier\nmain,barrier,Qubit

Finally, you can visualize and save the timing:

In [8]:
from qiskit_ibm_runtime.visualization import draw_circuit_schedule_timing

circuit_schedule = result[0].metadata["compilation"]["scheduler_timing"][
    "timing"
]
fig = draw_circuit_schedule_timing(
    circuit_schedule=circuit_schedule,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)

# Uncomment the following line to display the figure
# fig.show(renderer="notebook")

# Save to a file
# fig.write_html("scheduler_timing.html")

Successivamente, ottieni il timing della pianificazione del circuito: